# 02. 데이터 전처리 (Preprocessing)

**실행 환경:** 로컬 VSCode  
**목적:** 텍스트 정제, 레이블 통합, 학습 데이터셋 생성

---

## 전처리 파이프라인

```
Raw Data → 텍스트 정제 → 레이블 매핑 → 데이터 통합 → Train/Val/Test 분할 → 저장
```

In [1]:
import os
import re
import json
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ⭐ 경로 수정
BASE_PATH = Path.home() / 'CIVILCOMPLAINT' / 'data' / 'raw'
OUTPUT_PATH = Path.home() / 'CIVILCOMPLAINT' / 'data' / 'processed'
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

CALLCENTER_PATH = BASE_PATH / 'callcenter_qa'
DIALOGUE_PATH = BASE_PATH / 'korean_dialogue'
LLM_PATH = BASE_PATH / 'llm_instruction'

print(f"경로 확인: {BASE_PATH.exists()}")

경로 확인: True


---
## 1. 텍스트 정제 함수

In [2]:
class TextPreprocessor:
    """한국어 민원 텍스트 전처리기"""
    
    def __init__(self):
        # 개인정보 패턴
        self.phone_pattern = re.compile(r'\d{2,3}[-.]?\d{3,4}[-.]?\d{4}')
        self.email_pattern = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')
        self.ssn_pattern = re.compile(r'\d{6}[-]?\d{7}')  # 주민번호
        self.card_pattern = re.compile(r'\d{4}[-]?\d{4}[-]?\d{4}[-]?\d{4}')  # 카드번호
        
        # 특수문자 패턴
        self.html_pattern = re.compile(r'<[^>]+>')
        self.url_pattern = re.compile(r'https?://\S+|www\.\S+')
        self.emoji_pattern = re.compile(
            "["
            u"\U0001F600-\U0001F64F"  # 이모티콘
            u"\U0001F300-\U0001F5FF"  # 기호
            u"\U0001F680-\U0001F6FF"  # 교통
            u"\U0001F1E0-\U0001F1FF"  # 국기
            "]+", 
            flags=re.UNICODE
        )
    
    def mask_personal_info(self, text):
        """개인정보 마스킹"""
        text = self.phone_pattern.sub('[전화번호]', text)
        text = self.email_pattern.sub('[이메일]', text)
        text = self.ssn_pattern.sub('[주민번호]', text)
        text = self.card_pattern.sub('[카드번호]', text)
        return text
    
    def clean_text(self, text):
        """텍스트 정제"""
        if pd.isna(text) or not isinstance(text, str):
            return ''
        
        # HTML 태그 제거
        text = self.html_pattern.sub('', text)
        
        # URL 제거
        text = self.url_pattern.sub('', text)
        
        # 이모지 제거
        text = self.emoji_pattern.sub('', text)
        
        # 개인정보 마스킹
        text = self.mask_personal_info(text)
        
        # 특수문자 정리 (한글, 영문, 숫자, 기본 문장부호만 유지)
        text = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?\-()\[\]]', ' ', text)
        
        # 연속 공백 정리
        text = re.sub(r'\s+', ' ', text)
        
        return text.strip()
    
    def is_valid_text(self, text, min_len=5, max_len=1000):
        """텍스트 유효성 검사"""
        if not text or not isinstance(text, str):
            return False
        length = len(text)
        return min_len <= length <= max_len

# 전처리기 인스턴스
preprocessor = TextPreprocessor()

# 테스트
test_text = "안녕하세요 010-1234-5678로 연락주세요 😀 <br>감사합니다"
print(f"원본: {test_text}")
print(f"정제: {preprocessor.clean_text(test_text)}")

원본: 안녕하세요 010-1234-5678로 연락주세요 😀 <br>감사합니다
정제: 안녕하세요 [전화번호]로 연락주세요 감사합니다


---
## 2. 데이터 로드 및 전처리

In [3]:
def load_callcenter_data(path):
    """콜센터 JSON 데이터 로드"""
    all_data = []
    json_files = list(path.rglob('*.json'))
    
    for file in tqdm(json_files, desc='Loading callcenter'):
        try:
            with open(file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, list):
                    all_data.extend(data)
                else:
                    all_data.append(data)
        except:
            pass
    
    return pd.DataFrame(all_data)

def load_dialogue_data(path):
    """대화 XLSX 데이터 로드"""
    all_dfs = []
    xlsx_files = list(path.rglob('*.xlsx'))
    
    for file in tqdm(xlsx_files, desc='Loading dialogue'):
        try:
            df = pd.read_excel(file)
            all_dfs.append(df)
        except:
            pass
    
    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    return pd.DataFrame()

# 데이터 로드
print("데이터 로드 중...")
df_callcenter = load_callcenter_data(CALLCENTER_PATH)
df_dialogue = load_dialogue_data(DIALOGUE_PATH)

print(f"\nCallcenter: {len(df_callcenter):,} rows")
print(f"Dialogue: {len(df_dialogue):,} rows")

데이터 로드 중...


Loading dialogue: 100%|██████████| 13/13 [00:09<00:00,  1.32it/s]


Callcenter: 2,003,458 rows
Dialogue: 90,413 rows


---
## 3. 분류 모델용 데이터셋 생성

통합 스키마: `{text, domain, category, intent, source}`

In [4]:
def process_callcenter_for_classification(df):
    """콜센터 데이터 → 분류용 포맷 변환 (벡터화 + 다중 컬럼)"""
    
    # Q(질문)만 필터링
    df_q = df[df['QA'] == 'Q'].copy()
    print(f"Q 필터링: {len(df_q):,}건")
    
    # 텍스트 컬럼 통합 (고객질문 + 고객답변)
    # 고객질문(요청)이 비어있으면 고객답변 사용
    df_q['고객질문(요청)'] = df_q['고객질문(요청)'].fillna('')
    df_q['고객답변'] = df_q['고객답변'].fillna('')
    
    df_q['text_raw'] = df_q.apply(
        lambda row: row['고객질문(요청)'] if len(str(row['고객질문(요청)'])) > 0 
                    else row['고객답변'], 
        axis=1
    )
    
    # 벡터화된 텍스트 정제
    print("텍스트 정제 중...")
    df_q['text'] = df_q['text_raw'].apply(preprocessor.clean_text)
    
    # 유효한 텍스트만 필터링 (길이 5 이상)
    df_q['text_len'] = df_q['text'].str.len()
    df_valid = df_q[df_q['text_len'] >= 5].copy()
    
    print(f"유효한 텍스트: {len(df_valid):,}건 ({len(df_valid)/len(df_q)*100:.1f}%)")
    
    # 필요한 컬럼만 선택 및 이름 변경
    result = df_valid[['text', '도메인', '카테고리', '고객의도']].copy()
    result.columns = ['text', 'domain', 'category', 'intent']
    result['source'] = 'callcenter'
    
    # 중복 제거
    before = len(result)
    result = result.drop_duplicates(subset=['text'])
    print(f"중복 제거: {before:,} → {len(result):,}건")
    
    return result

# 콜센터 데이터 처리
df_callcenter_processed = process_callcenter_for_classification(df_callcenter)
print(f"\n처리된 콜센터 데이터: {len(df_callcenter_processed):,} rows")

Q 필터링: 1,014,311건
텍스트 정제 중...
유효한 텍스트: 387,995건 (38.3%)
중복 제거: 387,995 → 305,555건

처리된 콜센터 데이터: 305,555 rows


In [5]:
def process_dialogue_for_classification(df):
    """대화 데이터 → 분류용 포맷 변환"""
    processed = []
    
    text_col = 'SENTENCE' if 'SENTENCE' in df.columns else None
    domain_col = 'DOMAIN' if 'DOMAIN' in df.columns else None
    category_col = 'CATEGORY' if 'CATEGORY' in df.columns else None
    intent_col = 'MAIN' if 'MAIN' in df.columns else None
    speaker_col = 'SPEAKER' if 'SPEAKER' in df.columns else None
    qa_col = 'QA' if 'QA' in df.columns else None
    
    if not text_col:
        print("텍스트 컬럼을 찾을 수 없습니다.")
        return pd.DataFrame()
    
    # 고객 발화(Q)만 필터링
    if qa_col:
        df = df[df[qa_col] == 'Q'].copy()
    elif speaker_col:
        df = df[df[speaker_col] == '고객'].copy()
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Processing dialogue'):
        text = preprocessor.clean_text(row.get(text_col, ''))
        
        if not preprocessor.is_valid_text(text):
            continue
        
        processed.append({
            'text': text,
            'domain': row.get(domain_col, 'unknown'),
            'category': row.get(category_col, 'unknown'),
            'intent': row.get(intent_col, 'unknown'),
            'source': 'dialogue'
        })
    
    return pd.DataFrame(processed)

# 대화 데이터 처리
df_dialogue_processed = process_dialogue_for_classification(df_dialogue)
print(f"처리된 대화 데이터: {len(df_dialogue_processed):,} rows")

Processing dialogue: 100%|██████████| 51065/51065 [00:03<00:00, 13740.02it/s]

처리된 대화 데이터: 50,970 rows


In [6]:
# 데이터 통합
df_classification = pd.concat([df_callcenter_processed, df_dialogue_processed], ignore_index=True)

# 중복 제거
before_dedup = len(df_classification)
df_classification = df_classification.drop_duplicates(subset=['text'])
after_dedup = len(df_classification)

print(f"통합 데이터: {before_dedup:,} → 중복 제거 후: {after_dedup:,}")
print(f"\n도메인 분포:")
print(df_classification['domain'].value_counts())

통합 데이터: 356,525 → 중복 제거 후: 352,280

도메인 분포:
domain
질병관리본부    95368
K쇼핑       87548
금융/보험     80685
다산콜센터     41954
의복의류점      7799
음식점        7688
소매         7628
생활서비스      6472
부동산업       4497
카페         3930
숙박         3438
학원         2741
관광여가오락     2245
부동산         287
Name: count, dtype: int64


---
## 4. 레이블 인코딩

In [7]:
from sklearn.preprocessing import LabelEncoder

# 레이블 인코더 생성 및 적용
label_encoders = {}

for col in ['domain', 'category', 'intent']:
    le = LabelEncoder()
    df_classification[f'{col}_id'] = le.fit_transform(df_classification[col].fillna('unknown'))
    label_encoders[col] = le
    print(f"{col}: {len(le.classes_)} classes")

# 레이블 매핑 저장
label_mapping = {
    col: {i: label for i, label in enumerate(le.classes_)}
    for col, le in label_encoders.items()
}

with open(OUTPUT_PATH / 'label_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(label_mapping, f, ensure_ascii=False, indent=2)

print(f"\n레이블 매핑 저장: {OUTPUT_PATH / 'label_mapping.json'}")

domain: 14 classes
category: 63 classes
intent: 35153 classes

레이블 매핑 저장: /Users/kuka/CIVILCOMPLAINT/data/processed/label_mapping.json


---
## 5. Train / Val / Test 분할

In [8]:
# Stratified Split (domain 기준)
# Train: 80%, Val: 10%, Test: 10%

# 먼저 Train과 나머지 분리
train_df, temp_df = train_test_split(
    df_classification,
    test_size=0.2,
    stratify=df_classification['domain_id'],
    random_state=42
)

# Val과 Test 분리
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['domain_id'],
    random_state=42
)

print(f"Train: {len(train_df):,} ({len(train_df)/len(df_classification)*100:.1f}%)")
print(f"Val:   {len(val_df):,} ({len(val_df)/len(df_classification)*100:.1f}%)")
print(f"Test:  {len(test_df):,} ({len(test_df)/len(df_classification)*100:.1f}%)")

Train: 281,824 (80.0%)
Val:   35,228 (10.0%)
Test:  35,228 (10.0%)


In [9]:
# 분류 데이터 저장 - Parquet
train_df.to_parquet(OUTPUT_PATH / 'train_classification.parquet', index=False)
val_df.to_parquet(OUTPUT_PATH / 'val_classification.parquet', index=False)
test_df.to_parquet(OUTPUT_PATH / 'test_classification.parquet', index=False)

# 레이블 인코더 저장 - Joblib
import joblib
joblib.dump(label_encoders, OUTPUT_PATH / 'label_encoders.joblib')

print(f"분류 데이터 저장 완료:")
print(f"  - train_classification.parquet ({len(train_df):,}건)")
print(f"  - val_classification.parquet ({len(val_df):,}건)")
print(f"  - test_classification.parquet ({len(test_df):,}건)")
print(f"  - label_encoders.joblib")

분류 데이터 저장 완료:
  - train_classification.parquet (281,824건)
  - val_classification.parquet (35,228건)
  - test_classification.parquet (35,228건)
  - label_encoders.joblib


---
## 6. RAG용 QA 데이터셋 생성

In [10]:
def create_qa_pairs(df):
    """콜센터 데이터에서 QA 쌍 추출 (순차 매칭 방식)"""
    qa_pairs = []
    
    # 컬럼 확인
    dialogue_id_col = '대화셋일련번호'
    turn_col = '문장번호'
    qa_col = 'QA'
    q_col = '고객질문(요청)'
    a_col = '상담사답변'
    domain_col = '도메인'
    category_col = '카테고리'
    
    # 필수 컬럼 체크
    required_cols = [dialogue_id_col, turn_col, qa_col]
    if not all(col in df.columns for col in required_cols):
        print(f"필수 컬럼 없음. 사용 가능: {df.columns.tolist()}")
        return pd.DataFrame()
    
    # 대화별로 처리
    for dialogue_id, group in tqdm(df.groupby(dialogue_id_col), desc='Creating QA pairs'):
        # 문장번호 순으로 정렬
        group = group.sort_values(turn_col).reset_index(drop=True)
        
        # 연속된 Q→A 쌍 찾기
        for i in range(len(group) - 1):
            current = group.iloc[i]
            next_row = group.iloc[i + 1]
            
            # Q 다음에 A가 오는 경우
            if current[qa_col] == 'Q' and next_row[qa_col] == 'A':
                q_text = preprocessor.clean_text(current.get(q_col, ''))
                
                # 상담사답변 우선, 없으면 다른 컬럼 확인
                a_text = preprocessor.clean_text(next_row.get(a_col, ''))
                if not a_text:
                    a_text = preprocessor.clean_text(next_row.get('상담사질문(요청)', ''))
                
                # 유효한 쌍만 추가
                if len(q_text) >= 5 and len(a_text) >= 5:
                    qa_pairs.append({
                        'question': q_text,
                        'answer': a_text,
                        'domain': current.get(domain_col, 'unknown'),
                        'category': current.get(category_col, 'unknown'),
                        'dialogue_id': dialogue_id
                    })
    
    # DataFrame 변환 및 중복 제거
    df_qa = pd.DataFrame(qa_pairs)
    
    if len(df_qa) > 0:
        before = len(df_qa)
        df_qa = df_qa.drop_duplicates(subset=['question', 'answer'])
        print(f"중복 제거: {before:,} → {len(df_qa):,}")
    
    return df_qa

# QA 쌍 생성
df_qa = create_qa_pairs(df_callcenter)
print(f"\n생성된 QA 쌍: {len(df_qa):,}")

Creating QA pairs: 100%|██████████| 78998/78998 [01:58<00:00, 668.51it/s] 


중복 제거: 214,339 → 176,544

생성된 QA 쌍: 176,544


---
## 7. LLM 파인튜닝용 데이터셋 생성

In [11]:
def load_llm_instructions(path, max_files=None):
    """LLM Instruction 데이터 로드 및 변환"""
    instructions = []
    json_files = list(path.rglob('*.json'))[:max_files]
    
    for file in tqdm(json_files, desc='Loading LLM instructions'):
        try:
            with open(file, 'r', encoding='utf-8') as f:
                raw_data = json.load(f)
                
                # ⭐ 리스트면 첫 번째 항목 추출
                if isinstance(raw_data, list):
                    data = raw_data[0] if raw_data else {}
                else:
                    data = raw_data
                
                # instructions 필드 파싱
                if 'instructions' in data:
                    for inst_group in data['instructions']:
                        tuning_type = inst_group.get('tuning_type', 'unknown')
                        
                        if 'data' in inst_group:
                            for item in inst_group['data']:
                                instruction = item.get('instruction', '')
                                input_text = item.get('input', '')
                                output_text = item.get('output', '')
                                
                                if instruction and output_text:
                                    instructions.append({
                                        'instruction': instruction,  # 일단 clean_text 없이
                                        'input': input_text,
                                        'output': output_text,
                                        'task_type': tuning_type,
                                        'category': data.get('consulting_category', 'unknown')
                                    })
        except Exception as e:
            pass
    
    return pd.DataFrame(instructions)

# 다시 로드
df_llm = load_llm_instructions(LLM_PATH, max_files=None)
print(f"\n로드된 LLM instruction: {len(df_llm):,}")

Loading LLM instructions: 100%|██████████| 119182/119182 [00:52<00:00, 2272.02it/s]



로드된 LLM instruction: 119,182


In [12]:
# QA + LLM 데이터 저장
# QA 데이터 (RAG용)
if len(df_qa) > 0:
    df_qa.to_parquet(OUTPUT_PATH / 'qa_pairs.parquet', index=False)
    
    qa_documents = []
    for _, row in df_qa.iterrows():
        qa_documents.append({
            'content': f"질문: {row['question']}\n답변: {row['answer']}",
            'metadata': {
                'domain': row['domain'],
                'category': row['category'],
                'type': 'qa_pair'
            }
        })
    
    with open(OUTPUT_PATH / 'qa_documents.json', 'w', encoding='utf-8') as f:
        json.dump(qa_documents, f, ensure_ascii=False, indent=2)
    
    print(f"QA 데이터 저장 완료:")
    print(f"  - qa_pairs.parquet ({len(df_qa):,}건)")
    print(f"  - qa_documents.json")

# LLM 데이터 (파인튜닝용)
if len(df_llm) > 0:
    train_llm, val_llm = train_test_split(df_llm, test_size=0.1, random_state=42)
    
    train_llm.to_parquet(OUTPUT_PATH / 'train_llm.parquet', index=False)
    val_llm.to_parquet(OUTPUT_PATH / 'val_llm.parquet', index=False)
    
    print(f"\nLLM 데이터 저장 완료:")
    print(f"  - train_llm.parquet ({len(train_llm):,}건)")
    print(f"  - val_llm.parquet ({len(val_llm):,}건)")

QA 데이터 저장 완료:
  - qa_pairs.parquet (176,544건)
  - qa_documents.json

LLM 데이터 저장 완료:
  - train_llm.parquet (107,263건)
  - val_llm.parquet (11,919건)


In [13]:
# LLM 데이터 유효성 검사 및 필터링
if len(df_llm) > 0:
    # 빈 값 제거
    df_llm = df_llm[
        (df_llm['instruction'].str.len() > 5) & 
        (df_llm['output'].str.len() > 5)
    ]
    
    print(f"필터링 후: {len(df_llm):,}")
    print(f"\nTask Type 분포:")
    print(df_llm['task_type'].value_counts())

필터링 후: 85,417

Task Type 분포:
task_type
요약      35848
분류      31779
질의응답    17790
Name: count, dtype: int64


In [14]:
# Alpaca 포맷으로 변환 (파인튜닝용)
def to_alpaca_format(row):
    """Alpaca 형식으로 변환"""
    if row['input']:
        return f"""### Instruction:
{row['instruction']}

### Input:
{row['input']}

### Response:
{row['output']}"""
    else:
        return f"""### Instruction:
{row['instruction']}

### Response:
{row['output']}"""

if len(df_llm) > 0:
    df_llm['alpaca_text'] = df_llm.apply(to_alpaca_format, axis=1)
    
    # 샘플 확인
    print("=== Alpaca 포맷 샘플 ===")
    print(df_llm['alpaca_text'].iloc[0][:500])

=== Alpaca 포맷 샘플 ===
### Instruction:
다음 상담의 주제는 "상품 및 서비스 일반", "주문/결제/입금 확인", "취소/반품/교환/환불/AS", "배송 문의", "회원 관리", "제휴", "이벤트/할인", "콘텐츠", "기타" 중 무엇일까요?

### Input:
고객: 안녕하세요. ▲/▲▲부터 ▲▲일까지 4박 오션뷰로 예약하고 싶은데, 코로나 때문에 여행이 금지되면 환불은 어떻게 되나요?
상담사: 안녕하세요! ▲/▲▲부터 ▲▲일까지 더블식스 디럭스스윗 오션뷰 4박(2+2) 가격은 $920입니다. 발리에서는 코로나로 인한 특별 규정이 없어서 호텔 규정이 적용돼요. 체크인 15일 전까지 취소하면 여행사 수수료 3만 원만 공제하고 환불 가능해요.
고객: 그럼 조식 포함인가요?
상담사: 네. 조식과 세금이 포함되어 있어요. 객실 체크는 예약 요청 시 실시간으로 확인됩니다.
고객: 결제 가능한 최소 금액은 얼마인가요?
상담사: 카드 결제도 가능해요. 국민, 현대, 삼성, 신한카드로 결제할 수 있고,


In [15]:
# LLM 데이터 저장
if len(df_llm) > 0:
    # Train/Val 분할 (90/10)
    train_llm, val_llm = train_test_split(df_llm, test_size=0.1, random_state=42)
    
    # CSV 저장
    train_llm.to_csv(OUTPUT_PATH / 'train_llm.csv', index=False, encoding='utf-8-sig')
    val_llm.to_csv(OUTPUT_PATH / 'val_llm.csv', index=False, encoding='utf-8-sig')
    
    # JSONL 저장 (파인튜닝 도구 호환)
    with open(OUTPUT_PATH / 'train_llm.jsonl', 'w', encoding='utf-8') as f:
        for _, row in train_llm.iterrows():
            f.write(json.dumps({
                'instruction': row['instruction'],
                'input': row['input'],
                'output': row['output']
            }, ensure_ascii=False) + '\n')
    
    print(f"LLM 데이터 저장 완료:")
    print(f"  - Train: {len(train_llm):,}")
    print(f"  - Val: {len(val_llm):,}")

LLM 데이터 저장 완료:
  - Train: 76,875
  - Val: 8,542


In [16]:
# 축소 원인 분석
q_only = df_callcenter[df_callcenter['QA'] == 'Q']
print(f"Q만 필터링: {len(q_only):,}")

# 텍스트 길이 분포
q_only['text_len'] = q_only['고객질문(요청)'].fillna('').str.len()
print(f"길이 0인 것: {(q_only['text_len'] == 0).sum():,}")
print(f"길이 1~4인 것: {((q_only['text_len'] > 0) & (q_only['text_len'] < 5)).sum():,}")
print(f"길이 5 이상: {(q_only['text_len'] >= 5).sum():,}")

Q만 필터링: 1,014,311
길이 0인 것: 623,191
길이 1~4인 것: 2,360
길이 5 이상: 388,760


In [18]:
# 길이 0인 Q 레코드 분석
q_empty = df_callcenter[(df_callcenter['QA'] == 'Q') & 
                         (df_callcenter['고객질문(요청)'].fillna('').str.len() == 0)]

print(f"=== 길이 0인 Q 레코드 분석 ({len(q_empty):,}건) ===\n")

# 화자 분포 확인
print("화자 분포:")
print(q_empty['화자'].value_counts())

# 다른 텍스트 컬럼에 데이터가 있는지 확인
print("\n각 컬럼별 비어있지 않은 비율:")
text_cols = ['고객질문(요청)', '고객답변', '상담사질문(요청)', '상담사답변']
for col in text_cols:
    if col in q_empty.columns:
        non_empty = (q_empty[col].fillna('').str.len() > 0).sum()
        print(f"  {col}: {non_empty:,}건 ({non_empty/len(q_empty)*100:.1f}%)")

# 샘플 5개 확인
print("\n샘플 레코드:")
sample_cols = ['화자', 'QA', '고객의도', '고객질문(요청)', '상담사질문(요청)']
print(q_empty[sample_cols].head(5).to_string())

=== 길이 0인 Q 레코드 분석 (623,191건) ===

화자 분포:
화자
상담사    623180
고객         11
Name: count, dtype: int64

각 컬럼별 비어있지 않은 비율:
  고객질문(요청): 0건 (0.0%)
  고객답변: 194건 (0.0%)
  상담사질문(요청): 623,181건 (100.0%)
  상담사답변: 755건 (0.1%)

샘플 레코드:
     화자 QA 고객의도 고객질문(요청)                                                          상담사질문(요청)
2   상담사  Q                               아 그러세요. 정보 확인 후에 도와 드리겠습니다. 성함하고 전화번호 말씀 부탁 드립니다. 
4   상담사  Q                                                     소중한 정보 확인 감사합니다. 어느 상품이십니까? 
6   상담사  Q                                          ㅇㅇㅇㅇ 에어쿠션 컴포트 여성화 블랙 이백사십 주문하신 거 확인됩니다.
8   상담사  Q                                                    실례지만 교환하시는 사유가 사이즈가 작으신 건가요? 
10  상담사  Q                그러면 고객님 블랙 이백사십오로 교환해 드리겠습니다. 배송 받으신 상품 택 제거하시거나 외부 착화는 안 하셨습니까? 


---
## 8. 전처리 결과 요약

In [17]:
# 최종 요약
summary = {
    'classification': {
        'train': len(train_df),
        'val': len(val_df),
        'test': len(test_df),
        'total': len(df_classification),
        'num_domains': df_classification['domain'].nunique(),
        'num_categories': df_classification['category'].nunique()
    },
    'qa_pairs': len(df_qa) if 'df_qa' in dir() else 0,
    'llm_instructions': {
        'train': len(train_llm) if 'train_llm' in dir() else 0,
        'val': len(val_llm) if 'val_llm' in dir() else 0
    }
}

print("=" * 50)
print("전처리 완료 요약")
print("=" * 50)
print(f"\n[분류 데이터]")
print(f"  Train: {summary['classification']['train']:,}")
print(f"  Val:   {summary['classification']['val']:,}")
print(f"  Test:  {summary['classification']['test']:,}")
print(f"  도메인 수: {summary['classification']['num_domains']}")
print(f"  카테고리 수: {summary['classification']['num_categories']}")
print(f"\n[RAG QA 쌍]: {summary['qa_pairs']:,}")
print(f"\n[LLM 파인튜닝 데이터]")
print(f"  Train: {summary['llm_instructions']['train']:,}")
print(f"  Val:   {summary['llm_instructions']['val']:,}")

# 요약 저장
with open(OUTPUT_PATH / 'preprocessing_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"\n저장 위치: {OUTPUT_PATH}")

전처리 완료 요약

[분류 데이터]
  Train: 281,824
  Val:   35,228
  Test:  35,228
  도메인 수: 14
  카테고리 수: 63

[RAG QA 쌍]: 176,544

[LLM 파인튜닝 데이터]
  Train: 76,875
  Val:   8,542

저장 위치: /Users/kuka/CIVILCOMPLAINT/data/processed


---
## 다음 단계

→ **03_classification_model.ipynb** (Kaggle GPU):
1. KoBERT/KoELECTRA 로드
2. 분류 모델 파인튜닝
3. MLFlow 실험 추적